## 实施隐藏层
在下面，您将实现通过4x3x2网络的前向传递，并为两层提供S型激活功能。<br>
要做的事情：
- 计算对隐藏层的输入。
- 计算隐藏层输出。
- 计算输入到输出层。
- 计算网络的输出。

In [1]:
import numpy as np

def sigmoid(x):
    return 1/(1+np.exp(-x))

# 网络规模
N_input = 4
N_hidden = 3
N_output = 2

np.random.seed(42)
# 制作一些数据
X = np.random.randn(4)

weights_input_to_hidden = np.random.normal(0, scale=0.1, size=(N_input, N_hidden))
weights_hidden_to_output = np.random.normal(0, scale=0.1, size=(N_hidden, N_output))


# TODO: 通过网络进行传递

hidden_layer_in = np.dot(X, weights_input_to_hidden)
hidden_layer_out = sigmoid(hidden_layer_in)

print('隐藏层 输出:')
print(hidden_layer_out)

output_layer_in = np.dot(hidden_layer_out, weights_hidden_to_output)
output_layer_out = sigmoid(output_layer_in)

print('输出层 输出:')
print(output_layer_out)

隐藏层 输出:
[0.41492192 0.42604313 0.5002434 ]
输出层 输出:
[0.49815196 0.48539772]


## 反向传播
现在我们来解决如何使多层神经网络学习的问题。 之前，我们看到了如何使用梯度下降来更新权重。 反向传播算法只是其中的一种扩展，它使用链规则来查找关于将输入层连接到隐藏层（对于两层网络）的权重方面的错误。

要使用梯度下降将权重更新为隐藏层，您需要知道每个隐藏单元对最终输出造成了多少误差。 由于层的输出是由层之间的权重决定的，因此由单位导致的误差将通过网络中的权重来缩放。 由于我们知道输出中的错误，因此可以使用权重向后处理隐藏层。

例如，在输出层中，每个输出单位$k$都有误差$\delta^o_k$。归因于隐藏单元$j$的误差就是输出误差，该误差由输出层和隐藏层（以及渐变）之间的权重进行缩放：
$$\delta^h_j = \Sigma W_{jk}\delta^o_k f'(h_j)$$
然后，梯度下降步骤与之前相同，只是存在新的误差：
$$\Delta w_{ij} = \eta\delta^h_j x_i$$
其中$\Delta \omega_{ij}$是输入和隐藏层之间的权重，$x_i$是输入单位值。 这种形式适用于许多层。 权重步长等于步长乘以图层的输出误差乘以该图层的输入值:
$$\Delta w_{pq} = \eta\delta_{output} V_{in}$$
在这里，您可以通过从较高层向后传播误差来获得输出误差$\delta_{output}$。 输入值$V_{in}$是图层的输入，例如隐藏图层激活到输出单元。

**在NumPy中实施**<br>
在下面，您将实现代码以为两组权重计算一个反向传播更新步骤。<br>
要做的事
- 计算网络的输出误差。
- 计算输出层的误差项。
- 使用反向传播来计算隐藏层的误差项。
- 计算由于通过网络传播错误而导致的权重变化（增量权重）。

In [2]:
import numpy as np

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

x = np.array([0.5, 0.1, -0.2])
target = 0.6
learnrate = 0.5

weights_input_hidden = np.array([[0.5, -0.6],
                                 [0.1, -0.2],
                                 [0.1, 0.7]])

weights_hidden_output = np.array([0.1, -0.3])

# 前传
hidden_layer_input = np.dot(x, weights_input_hidden)
hidden_layer_output = sigmoid(hidden_layer_input)

output_layer_in = np.dot(hidden_layer_output, weights_hidden_output)
output = sigmoid(output_layer_in)

# 后传
# TODO: 计算输出误差
error = target - output

# TODO: 计算输出层的误差项
output_error_term = error * output * (1 - output)

# TODO: 计算隐藏层的误差项
hidden_error_term = np.dot(output_error_term, weights_hidden_output) * \
                    hidden_layer_output * (1 - hidden_layer_output)

# TODO: 计算隐藏层到输出层的权重变化
delta_w_h_o = learnrate * output_error_term * hidden_layer_output

# TODO: 计算输入层到隐藏层的权重变化
delta_w_i_h = learnrate * hidden_error_term * x[:, None]

print('隐藏层到输出层的权重的变化:')
print(delta_w_h_o)
print('输入层到隐藏层的权重的变化:')
print(delta_w_i_h)

隐藏层到输出层的权重的变化:
[0.00804047 0.00555918]
输入层到隐藏层的权重的变化:
[[ 1.77005547e-04 -5.11178506e-04]
 [ 3.54011093e-05 -1.02235701e-04]
 [-7.08022187e-05  2.04471402e-04]]


## 实施反向传播
现在我们已经看到输出层的误差项是
$$\delta_k = (y_k - \hat{y}_k)f'(a_k)$$
隐藏层的误差项是
$$\delta_j = \Sigma [w_{jk} \delta_k]f'(h_j)$$

现在，我们仅考虑一个具有一个隐藏层和一个输出单元的简单网络。 这是通过反向传播更新权重的一般算法：
- 将每一层的权重步长设置为零
 - 输入层到隐藏层的权重 $\Delta w_{ij} = 0$
 - 隐藏层到输出层的权重 $\Delta W_j = 0$
- 对于训练数据中的每个记录：
 - 通过网络进行前向传递，计算输出 $\hat{y}$
 - 计算输出单位中的误差梯度，$\delta^o = (y-\hat{y})f'(z)$，输入到输出单元$z=\Sigma_j W_j a_j$
 - 将误差传播到隐藏层 $\delta^h_j = \delta^o W_j f'(h_j)$
 - 更新权重步骤：
  - $\Delta W_j = \Delta W_j + \delta^o a_j$
  - $\Delta w_{ij} = \Delta w_{ij} + \delta^h_j a_j$
- 更新权重，其中$\eta$是学习率，$m$是记录数：
 - $W_j = W_j + \eta \Delta W_j / m$
 - $w_{ij} = w_{ij} + \eta \Delta w_{ij} / m$
- 重复进行迭代。

## 反向传播练习
目标：
- 实现前向通过。
- 实现反向传播算法。
- 更新权重。

In [6]:
import numpy as np
# 引入函数
from data_prep import features, targets, features_test, targets_test

np.random.seed(21)

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

# 超参数
n_hidden = 2  # 隐藏单位数
epochs = 900
learnrate = 0.005

n_records, n_features = features.shape
last_loss = None
# 初始化权重
weights_input_hidden = np.random.normal(scale=1 / n_features ** .5,
                                        size=(n_features, n_hidden))
weights_hidden_output = np.random.normal(scale=1 / n_features ** .5,
                                         size=n_hidden)

for e in range(epochs):
    del_w_input_hidden = np.zeros(weights_input_hidden.shape)
    del_w_hidden_output = np.zeros(weights_hidden_output.shape)
    for x, y in zip(features.values, targets):
        ### 前传 ###
        # TODO: 计算输出
        hidden_input = np.dot(x, weights_input_hidden)
        hidden_output = sigmoid(hidden_input)

        output = sigmoid(np.dot(hidden_output,
                                weights_hidden_output))

        ### 后传 ###
        # TODO: 计算网络的预测误差
        error = y - output

        # TODO: 计算输出单元的误差项
        output_error_term = error * output * (1 - output)

        ## 将误差传播到隐藏层

        # TODO: 计算隐藏层对误差的贡献
        hidden_error = np.dot(output_error_term, weights_hidden_output)

        # TODO: 计算隐藏层的误差项
        hidden_error_term = hidden_error * hidden_output * (1 - hidden_output)

        # TODO: 更新权重变化
        del_w_hidden_output += output_error_term * hidden_output
        del_w_input_hidden += hidden_error_term * x[:, None]

    # TODO: 更新权重
    weights_input_hidden += learnrate * del_w_input_hidden / n_records
    weights_hidden_output += learnrate * del_w_hidden_output / n_records

    # 在训练集上打印均方误差
    if e % (epochs / 10) == 0:
        hidden_output = sigmoid(np.dot(x, weights_input_hidden))
        out = sigmoid(np.dot(hidden_output,
                             weights_hidden_output))
        loss = np.mean((out - targets) ** 2)

        if last_loss and last_loss < loss:
            print("训练损失: ", loss, "  WARNING - 损失增加")
        else:
            print("训练损失: ", loss)
        last_loss = loss

# 计算测试数据的准确性
hidden = sigmoid(np.dot(features_test, weights_input_hidden))
out = sigmoid(np.dot(hidden, weights_hidden_output))
predictions = out > 0.5
accuracy = np.mean(predictions == targets_test)
print("预测精度:  {:.3f}".format(accuracy))

训练损失:  0.2513572524259881
训练损失:  0.24996540718842905
训练损失:  0.24862005218904504
训练损失:  0.2473199321717981
训练损失:  0.24606380465584854
训练损失:  0.24485044179257037
训练损失:  0.243678632018683
训练损失:  0.24254718151769472
训练损失:  0.24145491550165454
训练损失:  0.24040067932493334
预测精度:  0.725
